In [ ]:
def update_weights(self, pde_loss, bc_loss, alpha=0.1):
    """
    Implements the Wang et al. gradient-based weight update.
    This balances the 'stiffness' of the PDE loss vs. the BC loss.
    """
    self.optimizer.zero_grad()
    pde_loss.backward(retain_graph=True)
    grads_pde = []
    for param in self.model.parameters():
        if param.grad is not None:
            grads_pde.append(torch.abs(param.grad).flatten())
    max_grad_pde = torch.max(torch.cat(grads_pde))

    self.optimizer.zero_grad()
    bc_loss.backward(retain_graph=True)
    grads_bc = []
    for param in self.model.parameters():
        if param.grad is not None:
            grads_bc.append(torch.abs(param.grad).flatten())
    mean_grad_bc = torch.mean(torch.cat(grads_bc))

    # If mean_grad_bc is 0, we avoid division by zero
    lambda_hat = max_grad_pde / (mean_grad_bc + 1e-8)

    self.lambda_bc = (1.0 - alpha) * self.lambda_bc + alpha * lambda_hat
    
    self.optimizer.zero_grad()